# DeBERTa-v3-large — Sarcasm Detector (Vast.ai Version · v2)
### Three accuracy improvements over v1

| Fix | Change |
|-----|--------|
| 1 | `MAX_LEN 128 → 256` |
| 2 | Layer-wise LR decay |
| 3 | `deberta-v3-base → deberta-v3-large` |

| Component | Value |
|-----------|-------|
| Model | `microsoft/deberta-v3-large` (400M params) |
| Dataset | `train-balanced-sarcasm.csv` (upload to `/workspace/data/`) |
| Checkpoints | saved to `/workspace/checkpoints/` |


> ⚠️ **VRAM requirements:** deberta-v3-large needs ~20–22GB with batch_size=8.  
> RTX 3090 (24GB) ✅ · A40/A6000 (48GB) ✅ · RTX 3080 (10GB) ❌ use base instead.  
> Upload `train-balanced-sarcasm.csv` to `/workspace/data/` before running.

## Cell 1 — Install Dependencies

**What this cell does:** Ensures every required library is present at the exact pinned version before any import is attempted.

**Why the hard uninstall + reinstall?** DeBERTa-v3-large depends on very tightly coupled versions of `transformers`, `tokenizers`, `sentencepiece`, and `protobuf`. Even a minor mismatch (e.g., `tokenizers==0.20` instead of `0.19.1`) produces silent tokenization bugs or hard crashes. The uninstall step clears any conflicting pre-installed versions on the Vast.ai image before the pinned versions are installed.

**Why `os.kill(os.getpid(), 9)` at the end?** This force-kills the Jupyter kernel, triggering an automatic restart. Without it, Python's module cache (`sys.modules`) would hold stale references to the old library versions — the newly installed packages would not be picked up until the next session.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
## PURPOSE: Ensure every required library is present at the exact pinned version
# needed by DeBERTa-v3-large before any import is attempted.
#
# WHY THE HARD UNINSTALL + REINSTALL?
#   DeBERTa relies on tightly coupled versions of `transformers`, `tokenizers`,
#   `sentencepiece`, and `protobuf`. A version mismatch causes silent tokenization
#   bugs or runtime crashes.
import importlib
import subprocess

def ensure_installed(packages: dict):
    """
    Check if packages are installed, install only the missing ones.

    packages: dict of {import_name: pip_install_name}
    e.g. {'sklearn': 'scikit-learn', 'pandas': 'pandas'}
    """
    missing = []
    for import_name, pip_name in packages.items():
        if importlib.util.find_spec(import_name) is None:
            print(f'  ✗ {import_name} — not found, will install')
            missing.append(pip_name)
        else:
            print(f'  ✓ {import_name} — already installed')

    if missing:
        print(f'\nInstalling: {missing}')
        subprocess.run(['pip', 'install', '-q'] + missing, check=True)
        print('Done.')
    else:
        print('\nAll packages already installed.')

# Usage
ensure_installed({
    'pandas':       'pandas',
    'numpy':        'numpy',
    'torch':        'torch',
    'datasets':     'datasets',
    'sklearn':      'scikit-learn',
    'transformers': 'transformers==4.44.2',
    'sentencepiece':'sentencepiece==0.1.99',
    'accelerate':   'accelerate',
})

import subprocess
subprocess.run(['pip', 'uninstall', '-y', 'transformers', 'tokenizers', 'protobuf', 'sentencepiece', 'wandb'],
               capture_output=True)
subprocess.run(['pip', 'install', '-q',
                'transformers==4.44.2',
                'tokenizers==0.19.1',
                'sentencepiece==0.1.99',
                'protobuf==3.20.3',
                'datasets',
                'scikit-learn',
                'accelerate',
               'pandas',
               'numpy'],
               capture_output=True)

import os
os.environ['WANDB_DISABLED'] = 'true'
print('Installation complete.')

import subprocess, sys, os

# Install using the kernel's own python
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'numpy', 'datasets', 'scikit-learn',
                'transformers==4.44.2', 'tokenizers==0.19.1',
                'sentencepiece==0.1.99', 'protobuf==3.20.3',
                'accelerate'], check=True)

print('Done — restarting kernel now...')
os.kill(os.getpid(), 9)  # force kernel restart

  ✗ pandas — not found, will install
  ✓ numpy — already installed
  ✓ torch — already installed
  ✗ datasets — not found, will install
  ✗ sklearn — not found, will install
  ✗ transformers — not found, will install
  ✗ sentencepiece — not found, will install
  ✗ accelerate — not found, will install

Installing: ['pandas', 'datasets', 'scikit-learn', 'transformers==4.44.2', 'sentencepiece==0.1.99', 'accelerate']


Done.
Installation complete.


## Cell 2 — Imports & GPU Check

**What this cell does:** Imports all project libraries and validates that the runtime has a GPU with sufficient VRAM *before* any model is loaded.

**Key imports explained:**
- `DebertaV2Tokenizer` / `DebertaV2ForSequenceClassification` — DeBERTa-specific classes; cannot be replaced by BERT equivalents (different tokenization scheme).
- `Trainer` + `TrainingArguments` — HuggingFace's high-level training loop that handles batching, gradient accumulation, checkpointing, and evaluation automatically.
- `EarlyStoppingCallback` — halts training when validation accuracy stops improving.
- `AdamW` (from `torch.optim`) — weight-decoupled Adam, the standard optimizer for transformer fine-tuning.

**Why the VRAM guard (`< 18 GB`)?** `deberta-v3-large` requires ~20–22 GB at `batch_size=8` with FP16. An under-sized GPU would OOM mid-epoch — better to catch this immediately than after 30 minutes of tokenization.

In [ ]:
# ── Cell 2: Imports & GPU check ────────────────────────────────────────────
#
# PURPOSE: Import all project dependencies and validate hardware
# before any expensive model loading begins.
#
# KEY IMPORTS:
#   DebertaV2Tokenizer / DebertaV2ForSequenceClassification
#     DeBERTa-specific classes; cannot be swapped for BERT equivalents.
#   Trainer + TrainingArguments
#     HuggingFace high-level loop: handles batching, checkpointing, evaluation.
#   EarlyStoppingCallback
#     Halts training when val accuracy stops improving (saves GPU cost).
#   AdamW (torch.optim)
#     Weight-decoupled Adam; standard optimizer for transformer fine-tuning.

import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    DebertaV2Tokenizer,
    DebertaV2ForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM     : {vram:.1f} GB')
    if vram < 18:
        print('WARNING: < 18GB VRAM — consider switching back to deberta-v3-base')
else:
    print('WARNING: No GPU detected! Training will be very slow.')

PyTorch  : 2.11.0+cu128
Device   : cuda
GPU      : NVIDIA A100-PCIE-40GB
VRAM     : 42.4 GB


Cell 3 — Paths & Configuration
What this cell does: Declares all global constants in one place — file paths, model name, and token budget.

Why centralise config? Keeping every tunable value here means you can swap dataset, model variant, or MAX_LEN without hunting through multiple cells. It also makes hyperparameter ablations easy to reproduce.

FIX 1 — MAX_LEN = 256: Reddit comments plus their parent posts can easily exceed 128 tokens. At 128, long parents were aggressively truncated, stripping the very contextual cues that signal sarcasm. Doubling the budget costs ~2× attention memory but recovers that context. Expected gain: +1–2% accuracy.

FIX 3 — deberta-v3-large: The large variant has 24 transformer layers and ~400 M parameters versus 12 layers / 183 M for the base model. More depth = more capacity to model the subtle tonal shifts that distinguish sarcasm.

In [ ]:
# ── Cell 3: Paths & config ─────────────────────────────────────────────────
#
# PURPOSE: Declare all global constants in one place — file paths, model name,
# and token budget — so the rest of the notebook contains no magic strings.

DATA_PATH  = '/workspace/train-balanced-sarcasm.csv'
CKPT_DIR   = '/workspace/checkpoints/deberta-large-sarcasm'
FINAL_DIR  = '/workspace/checkpoints/deberta-large-sarcasm-final'

# ── deberta-v3-large instead of deberta-v3-base(400M params) ────────────────────
# 24 transformer layers vs 12 — more capacity to model subtle tonal cues for sarcasm.
MODEL_NAME = 'microsoft/deberta-v3-large'

# ── MAX_LEN = 256 ──────────────────────────────────────────────
# Reddit comments + parent posts often exceed 128 tokens. At 128, long parents were
# aggressively truncated, losing the contextual contrast that signals sarcasm.
# Doubling to 256 costs ~2× attention memory but recovers that critical context.
MAX_LEN    = 256

# exist_ok=True prevents errors if the directories already exist on re-runs.
os.makedirs('/workspace/data', exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

# Fail fast with a helpful message rather than a cryptic pandas error in the next cell.
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Dataset not found at {DATA_PATH}\n'
        'Please upload train-balanced-sarcasm.csv to /workspace/data/ '
        'using the Jupyter file browser on the left.'
    )
print(f'Model      : {MODEL_NAME}')
print(f'MAX_LEN    : {MAX_LEN}')
print(f'Dataset    : {DATA_PATH}')
print(f'Checkpoints: {CKPT_DIR}')

Model      : microsoft/deberta-v3-large
MAX_LEN    : 256
Dataset    : /workspace/train-balanced-sarcasm.csv
Checkpoints: /workspace/checkpoints/deberta-large-sarcasm


## Cell 4 — Load & Preprocess Data

**What this cell does:** Loads the SARC CSV and engineers the input text the model will receive during training.

**Why `comment [SEP] parent_comment`?**
DeBERTa supports two-segment inputs separated by `[SEP]`. The comment comes first because the label applies to it — placing it first means it is less likely to be truncated when the combined text exceeds `MAX_LEN`. The parent provides tonal contrast: sarcasm is often only recognisable relative to what it replies to.

**Why `fillna('')` instead of `dropna()`?**
Top-level Reddit posts have no parent comment. Dropping those rows would lose a large fraction of the dataset. An empty string is the correct neutral value — the model simply attends to nothing after `[SEP]`.

In [ ]:
# ── Cell 4: Load & preprocess data ─────────────────────────────────────────
#
# PURPOSE: Load the raw SARC CSV and engineer the input text that the model sees.
#
# TEXT FORMAT — why 'comment [SEP] parent_comment'?
#   DeBERTa supports two-segment inputs separated by [SEP].
#   The comment comes FIRST because the label applies to it — it is less likely
#   to be truncated if the combined text exceeds MAX_LEN.
#   The parent provides tonal contrast: sarcasm is often only recognisable
#   relative to the literal meaning of what it replies to.

print('Loading dataset...')
# Drop rows where 'comment' is NaN — there is nothing to classify.
df = pd.read_csv(DATA_PATH).dropna(subset=['comment'])

# Fill missing parent_comment with '' rather than dropping the row.
# Top-level Reddit posts have no parent, but the comment alone still carries signal.
df['parent_comment'] = df['parent_comment'].fillna('')

# Keep only the three columns we need (drops URL, author, subreddit, etc.)
df = df[['comment', 'parent_comment', 'label']]

# Build the combined input: comment first, then [SEP] separator, then parent context.
df['text'] = df['comment'] + ' [SEP] ' + df['parent_comment']
df = df[['text', 'label']]

print(f'Total samples : {len(df):,}')
print(df['label'].value_counts())

Loading dataset...
Total samples : 1,010,771
label
0    505403
1    505368
Name: count, dtype: int64


## Cell 5 — Train / Val / Test Split

**What this cell does:** Partitions data into three non-overlapping sets:
- **Train (80%)** — used for gradient updates
- **Val (10%)** — used for early stopping and checkpoint selection
- **Test (10%)** — used *only* for the final unbiased evaluation

**Why stratify?** Stratification preserves the original 50/50 class ratio in each split. Without it, random chance could skew one split toward more sarcastic examples, making val/test metrics unreliable.

**Why `reset_index(drop=True)`?** After slicing a DataFrame the original row indices are retained. `Dataset.from_pandas` may expose those as an unwanted column (`__index_level_0__`) that gets passed to the model as an extra input. `reset_index(drop=True)` produces a clean 0…N index and avoids this.

In [ ]:
# ── Cell 5: Train / val / test split ───────────────────────────────────────
#
# PURPOSE: Partition data into three non-overlapping sets:
#   Train (80%) – gradient updates
#   Val   (10%) – early stopping and checkpoint selection
#   Test  (10%) – final unbiased evaluation only (never seen during training)
#
# WHY STRATIFY?
#   Preserves the 50/50 class ratio in each split. Without it, random chance
#   could skew one split toward more sarcastic examples, making metrics unreliable.
#
# WHY reset_index(drop=True)?
#   After slicing a DataFrame, original row indices are retained. Dataset.from_pandas
#   may expose them as '__index_level_0__', which Trainer passes to the model as an
#   unwanted extra input. reset_index(drop=True) produces a clean 0…N index.

# First split: 80% train, 20% temp
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
# Second split: divide the 20% equally → 10% val, 10% test
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f'Train : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}')

# Convert to HuggingFace Dataset objects — required by Trainer.train() / predict()
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))

Train : 808,616  |  Val : 101,077  |  Test : 101,078


## Cell 6 — Tokenizer

**What this cell does:** Converts raw text strings into fixed-length integer token ID sequences that the model can process as tensors.

**Why `DebertaV2Tokenizer` specifically?**
DeBERTa uses SentencePiece (unigram language model) tokenization with a 128 k vocabulary — different from BERT's WordPiece. Using the wrong tokenizer would produce incorrect token IDs and break the embedding lookup entirely.

**Tokenization parameters:**
- `padding='max_length'` — pads short sequences to `MAX_LEN=256` so all tensors in a batch are the same shape (required for GPU matrix operations).
- `truncation=True` — silently drops tokens beyond `MAX_LEN` rather than raising an error.
- `max_length=MAX_LEN` — **FIX 1**: 256 instead of 128.

**Why `batched=True` in `.map()`?** Processes rows in chunks of 1 000 instead of one at a time, which is ~10–50× faster on a dataset of 1 M rows.

In [ ]:
# ── Cell 6: Tokenizer ──────────────────────────────────────────────────────
#
# PURPOSE: Convert raw text strings into fixed-length integer token ID sequences
# that the model processes as tensors.
#
# WHY DebertaV2Tokenizer specifically?
#   DeBERTa uses SentencePiece (unigram LM) with a 128k vocabulary — different
#   from BERT's WordPiece. Using the wrong tokenizer produces incorrect token IDs
#   and breaks the embedding lookup entirely.
#
# TOKENIZATION PARAMS:
#   padding='max_length' → pads short sequences to MAX_LEN so all batch tensors
#                          are the same shape (required for GPU matrix ops).
#   truncation=True      → silently drops tokens beyond MAX_LEN.
#   max_length=MAX_LEN

print(f'Loading tokenizer: {MODEL_NAME}')
# from_pretrained downloads tokenizer_config.json + spm.model (SentencePiece vocab)
tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)
print(f'Vocab size : {tokenizer.vocab_size:,}')

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
    )

# batched=True processes rows in chunks of 1000 — ~10-50x faster than row-by-row
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)
print('Tokenization complete.')

Loading tokenizer: microsoft/deberta-v3-large


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Vocab size : 128,000


Map:   0%|          | 0/808616 [00:00<?, ? examples/s]

Map:   0%|          | 0/101077 [00:00<?, ? examples/s]

Map:   0%|          | 0/101078 [00:00<?, ? examples/s]

Tokenization complete.


## Cell 7 — Load Model

**What this cell does:** Downloads the pretrained `deberta-v3-large` weights and attaches a fresh binary classification head for sarcasm detection.

**How `num_labels=2` works:**
The pretrained checkpoint was trained for masked language modelling (MLM), so its output head predicts over the full 128 k vocabulary. Setting `num_labels=2` discards that MLM head and replaces it with a new linear layer: `hidden_dim → 2 logits` (not_sarcastic, sarcastic). The new head is randomly initialised — it learns entirely during fine-tuning. The 24 transformer layers keep their pretrained weights.

**The warning `Some weights … are newly initialized` is expected** — it refers precisely to `classifier.weight/bias` and `pooler.dense.weight/bias`, which are the new head parameters described above.

In [ ]:
# ── Cell 7: Load model ─────────────────────────────────────────────────────
#
# PURPOSE: Download pretrained DeBERTa-v3-large weights and attach a fresh
# binary classification head.
#
# deberta-v3-large has 24 transformer layers (~400M params)
# vs 12 layers / 183M for base — more depth to model long-range sarcasm signals.
#
# HOW num_labels=2 WORKS:
#   The pretrained checkpoint's output head predicts over the full 128k vocabulary
#   (masked language modelling). num_labels=2 discards that MLM head and replaces
#   it with a new linear layer: hidden_dim → 2 logits (not_sarcastic, sarcastic).
#   The new head is randomly initialised; the 24 transformer layers keep pretrained weights.
#   The warning 'Some weights … are newly initialized' is EXPECTED and refers to
#   classifier.weight/bias and pooler.dense.weight/bias.

# Loading deberta-v3-large (~400M params vs 183M for base)
print(f'Loading model: {MODEL_NAME}')
model = DebertaV2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2, # binary classification: 0=not_sarcastic, 1=sarcastic
)

# Audit parameter counts — for deberta-v3-large both values should be ~435M.
# If trainable < total, some layers are frozen (not the case here — all are trained)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable:,}')

Loading model: microsoft/deberta-v3-large


pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters    : 435,063,810
Trainable parameters: 435,063,810


## Cell 8 — Metrics Function

**What this cell does:** Defines `compute_metrics()`, which the Trainer calls after every epoch to evaluate the model on the validation set.

**Why both accuracy and F1?**
- *Accuracy*: overall fraction correct — intuitive and directly comparable to prior work on SARC.
- *Weighted F1*: accounts for per-class precision/recall balance. Even on a balanced dataset, F1 reveals if the model is secretly biased toward one class (e.g., high recall on sarcastic but low precision).

**How it connects to Trainer:** `metric_for_best_model='accuracy'` in Cell 9 uses the `'accuracy'` key returned here to decide which epoch's checkpoint to keep.

In [ ]:
# ── Cell 8: Metrics ────────────────────────────────────────────────────────
#
# PURPOSE: Define the evaluation function that Trainer calls after every epoch.
#
# WHY BOTH accuracy AND f1?
#   accuracy – overall fraction correct; intuitive and directly comparable to
#              prior work on the SARC dataset.
#   weighted F1 – reveals per-class precision/recall balance. Even on a 50/50
#                 dataset, F1 can expose bias (e.g., high recall on sarcastic
#                 but low precision due to over-predicting that class).
#
# HOW IT CONNECTS TO TRAINER:
#   metric_for_best_model='accuracy' (Cell 9) uses the 'accuracy' key returned
#   here to decide which epoch's checkpoint to save as the best.

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }

## Cell 9 — Training Arguments

**What this cell does:** Configures every hyperparameter of the training loop.

**Batch size strategy — why gradient accumulation?**
`deberta-v3-large` is ~2× the size of the base model and needs a smaller per-device batch to fit in VRAM. `gradient_accumulation_steps=2` accumulates gradients over 2 mini-batches before an optimizer step, simulating an effective batch of `16 × 2 = 32` without exceeding VRAM.

**Key hyperparameters:**
- `warmup_ratio=0.15` — LR ramps from 0 to peak over the first 15 % of steps, preventing destructive updates to pretrained weights in the early steps.
- `weight_decay=0.01` — L2 regularisation on weight matrices to reduce overfitting.
- `load_best_model_at_end=True` — restores the best-accuracy checkpoint even if the final epoch overfits.
- `fp16=True` (on CUDA) — mixed-precision halves VRAM usage and speeds up Tensor Core ops.
- `dataloader_num_workers=4` + `dataloader_pin_memory=True` — prevents the CPU data pipeline from bottlenecking the GPU (without this, GPU utilisation can drop to ~30 %).

In [ ]:
# ── Cell 9: Training arguments ─────────────────────────────────────────────
#
# PURPOSE: Configure every hyperparameter of the training loop in one place.
#
# BATCH SIZE STRATEGY — gradient accumulation explained:
#   deberta-v3-large is ~2× the size of base, so the per-device batch must be
#   smaller to fit in VRAM. gradient_accumulation_steps compensates by accumulating
#   gradients over GRAD_ACCUM mini-batches before an optimizer step, simulating
#   a larger effective batch without exceeding VRAM.
#
#   GPU VRAM     | TRAIN_BATCH | GRAD_ACCUM | Effective batch
#   RTX 3090 24G |      8      |     4      |       32
#   A40/A6000 48G|     16      |     2      |       32  ← current config
#   A100 80GB    |     32      |     1      |       32


TRAIN_BATCH   = 16    # per-device batch size (tune to your GPU VRAM)
GRAD_ACCUM    = 2     # accumulate gradients over this many steps before optimizer step
LEARNING_RATE = 2e-5  # peak LR for the top transformer layer and classifier head
NUM_EPOCHS    = 5     # upper bound; early stopping may terminate training sooner

args = TrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = TRAIN_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    per_device_eval_batch_size  = 16,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.15,           # LR ramps from 0→peak over first 15% of steps
    weight_decay                = 0.01,           # L2 regularisation on weight matrices
    evaluation_strategy               = 'epoch',  # evaluate on val set after every epoch
    save_strategy               = 'epoch',        # save a checkpoint after every epoch
    load_best_model_at_end      = True,           # restore the best checkpoint at the end
    metric_for_best_model       = 'accuracy',     # 'accuracy' key from compute_metrics
    greater_is_better           = True,           # higher accuracy = better
    logging_steps               = 200,            # log train loss every 200 optimizer steps
    fp16                        = torch.cuda.is_available(),  # mixed-precision on GPU
    report_to                   = 'none',         # disable WandB / TensorBoard
    dataloader_num_workers = 4,   # parallel CPU data loading; prevents GPU starvation
    dataloader_pin_memory  = True, # pin tensors in page-locked memory for faster CPU→GPU transfer
)

print('Training arguments set.')
print(f'  batch_size (per device)  : {args.per_device_train_batch_size}')
print(f'  gradient_accumulation    : {args.gradient_accumulation_steps}')
print(f'  effective batch_size     : {TRAIN_BATCH * GRAD_ACCUM}')
print(f'  learning_rate            : {args.learning_rate}')
print(f'  fp16                     : {args.fp16}')

Training arguments set.
  batch_size (per device)  : 16
  gradient_accumulation    : 2
  effective batch_size     : 32
  learning_rate            : 2e-05
  fp16                     : True


/venv/main/lib/python3.12/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Cell 10 — Layer-wise Learning Rate Decay (FIX 2)

**What this cell does:** Assigns a different learning rate to each transformer layer, decaying exponentially from the top layer (full LR) down to the embeddings (lowest LR).

**Why?** Lower transformer layers encode general linguistic knowledge (syntax, morphology) that was expensively learned during pretraining. Updating them aggressively would cause catastrophic forgetting. Upper layers encode task-specific features and need faster adaptation.

**Decay formula:**  
`layer_lr(i) = LEARNING_RATE × DECAY_RATE^(num_layers − 1 − i)`  
With `DECAY_RATE=0.9` and 24 layers:
- Top layer (i=23): `2e-5 × 0.9⁰ = 2.00e-05`
- Bottom layer (i=0): `2e-5 × 0.9²³ ≈ 1.78e-06`
- Embeddings: `2e-5 × 0.9²⁴ ≈ 1.60e-06`

**Why exclude bias / LayerNorm from weight decay?**
These parameters do not benefit from L2 regularisation. Applying weight decay to biases in particular has been shown empirically to hurt performance.

**Scheduler:** Linear warmup (15 % of steps) followed by linear decay to 0 — the standard schedule from the original BERT fine-tuning recipe.

In [ ]:
# ── Cell 10: FIX 2 — Layer-wise learning rate decay ────────────────────────
#
# PURPOSE: Assign a different learning rate to each transformer layer, decaying
# exponentially from the top layer (full LR) down to the embeddings (lowest LR).
#
# MOTIVATION:
#   Lower layers encode general linguistic knowledge (syntax, morphology) that was
#   expensively learned during pretraining. Large LR updates would cause catastrophic
#   forgetting. Upper layers encode task-specific features and need faster adaptation.
#
# DECAY FORMULA:
#   layer_lr(i) = LEARNING_RATE * DECAY_RATE^(num_layers - 1 - i)
#   i=0 (bottom): smallest LR.  i=num_layers-1 (top): full LR.
#
# NO_DECAY SET:
#   Bias terms and LayerNorm params are excluded from weight decay.
#   Applying L2 regularisation to biases hurts performance empirically.

DECAY_RATE  = 0.9   # LR multiplier per layer going downward

# Read layer count from model config (don't hard-code 24 — works for other variants too)
num_layers = model.config.num_hidden_layers
print(f'Number of hidden layers: {num_layers}')

# Parameters that should NOT receive weight decay (standard practice for transformers)
no_decay = {'bias', 'LayerNorm.weight', 'LayerNorm.bias'}

optimizer_grouped_parameters = []

# ── Embedding layer — lowest LR ────────────────────────────────────────────
# Embeddings encode the most general word-level semantics; they need the smallest LR.
embed_lr = LEARNING_RATE * (DECAY_RATE ** num_layers)
optimizer_grouped_parameters += [
    {   # weight matrices in embeddings → apply weight decay
        'params': [p for n, p in model.named_parameters()
                   if 'embeddings' in n and not any(nd in n for nd in no_decay)],
        'lr': embed_lr, 'weight_decay': 0.01,
    },
    {   # biases and LayerNorm in embeddings → no weight decay
        'params': [p for n, p in model.named_parameters()
                   if 'embeddings' in n and any(nd in n for nd in no_decay)],
        'lr': embed_lr, 'weight_decay': 0.0,
    },
]

# ── One parameter group per transformer layer ───────────────────────────────
# Layer 0 (bottom) = smallest LR; layer num_layers-1 (top) = largest LR.
for layer_idx in range(num_layers):
    # Layer 0 (bottom) gets the smallest LR, layer N-1 (top) gets the largest
    layer_lr = LEARNING_RATE * (DECAY_RATE ** (num_layers - 1 - layer_idx))
    optimizer_grouped_parameters += [
        {
            'params': [p for n, p in model.named_parameters()
                       if f'layer.{layer_idx}.' in n
                       and not any(nd in n for nd in no_decay)],
            'lr': layer_lr, 'weight_decay': 0.01,
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if f'layer.{layer_idx}.' in n
                       and any(nd in n for nd in no_decay)],
            'lr': layer_lr, 'weight_decay': 0.0,
        },
    ]

# ── Classifier head & pooler — full LR ─────────────────────────────────────
# These weights are newly initialised (Cell 7) and need the highest LR to adapt
# quickly from random initialisation to the sarcasm task.
optimizer_grouped_parameters += [
    {
        'params': [p for n, p in model.named_parameters()
                   if ('classifier' in n or 'pooler' in n)
                   and not any(nd in n for nd in no_decay)],
        'lr': LEARNING_RATE, 'weight_decay': 0.01,
    },
    {
        'params': [p for n, p in model.named_parameters()
                   if ('classifier' in n or 'pooler' in n)
                   and any(nd in n for nd in no_decay)],
        'lr': LEARNING_RATE, 'weight_decay': 0.0,
    },
]

# Construct AdamW with the layered parameter groups defined above.
# Each group carries its own 'lr' and 'weight_decay', overriding AdamW defaults.
optimizer = AdamW(optimizer_grouped_parameters)

# ── LR Scheduler: linear warmup → linear decay ─────────────────────────────
# Total optimizer steps = (batches per epoch) × epochs
num_training_steps = (
    len(train_dataset) // (TRAIN_BATCH * GRAD_ACCUM) + 1
) * NUM_EPOCHS
num_warmup_steps   = int(num_training_steps * 0.15)  # first 15% of steps ramp LR up

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = num_warmup_steps,
    num_training_steps = num_training_steps,
)

print(f'Layer-wise LR decay built.')
print(f'  Embedding LR : {embed_lr:.2e}')
print(f'  Top layer LR : {LEARNING_RATE:.2e}')
print(f'  Decay factor : {DECAY_RATE} per layer')
print(f'  Warmup steps : {num_warmup_steps} / {num_training_steps}')


Number of hidden layers: 24
Layer-wise LR decay built.
  Embedding LR : 1.60e-06
  Top layer LR : 2.00e-05
  Decay factor : 0.9 per layer
  Warmup steps : 18952 / 126350


## Throughput Diagnostic Cell

**What this cell does:** Computes effective training throughput in samples/sec and reminds the user to check GPU utilisation.

**How to interpret the output:**
- `Effective samples/sec = (iterations/sec) × batch_size × grad_accumulation`
- On a healthy A100/A40 you should see 200+ samples/sec.
- Low throughput (< 100) usually means the CPU data pipeline is the bottleneck — increase `dataloader_num_workers` or verify the tokenized dataset is cached.
- Replace `1.24` with your actual `it/s` value from the Trainer progress bar.

In [ ]:
# ── Throughput Diagnostic ──────────────────────────────────────────────────
#
# PURPOSE: Estimate effective GPU throughput (samples/sec) to catch data
# pipeline bottlenecks before committing to a full multi-hour training run.
#
# HOW TO READ:
#   Effective samples/sec = (it/s from Trainer progress bar) × batch × accum
#   On a healthy A100/A40: expect 200+ samples/sec.
#   Low values (<100) typically mean:
#     (a) CPU data loading is the bottleneck → increase dataloader_num_workers
#     (b) GPU is thermal/power throttling → check nvidia-smi
#     (c) Tokenized dataset is not cached → re-run .map() with a cache dir
#
# Replace 1.24 with your actual it/s observed from the Trainer progress bar.
print(f"Effective samples/sec: {1.24 * TRAIN_BATCH * GRAD_ACCUM:.0f}")
print(f"GPU utilization — run nvidia-smi in terminal to check")


Effective samples/sec: 40
GPU utilization — run nvidia-smi in terminal to check


## Cell 11 — Train

**What this cell does:** Instantiates the `Trainer` and launches the full training loop.

**`optimizers=(optimizer, scheduler)` — FIX 2 in action:**
Passing this tuple overrides Trainer's default AdamW and activates the layer-wise LR decay built in Cell 10. Without this argument, all layers would receive the same LR.

**`EarlyStoppingCallback(patience=3)`:**
If validation accuracy does not improve for 3 consecutive epochs, training stops. Combined with `load_best_model_at_end=True`, the saved model is always from the epoch with the highest validation accuracy.

**What `trainer.train()` does each epoch:**
1. Iterate over train batches → forward pass → cross-entropy loss
2. Backward pass → accumulate gradients for `GRAD_ACCUM` steps
3. Optimizer step → update weights using layer-wise LRs
4. Scheduler step → advance LR along warmup/decay curve
5. Evaluate on val set → compute accuracy + F1

In [ ]:
# ── Cell 11: Train ─────────────────────────────────────────────────────────
#
# PURPOSE: Instantiate the Trainer and run the full training loop.
#
# KEY DESIGN DECISIONS:
#
# optimizers=(optimizer, scheduler)  ← FIX 2
#   Passing our custom (optimizer, scheduler) tuple overrides Trainer's default
#   AdamW setup and activates the layer-wise LR decay built in Cell 10.
#   Without this argument, all layers would receive the same LR.
#
# EarlyStoppingCallback(patience=3)
#   If val accuracy does not improve for 3 consecutive epochs, training stops.
#   Combined with load_best_model_at_end=True, the final saved model is always
#   from the epoch with the highest validation accuracy.
#
# WHAT TRAINER.TRAIN() DOES EACH EPOCH:
#   1. Iterate over train_dataset in TRAIN_BATCH mini-batches
#   2. Forward pass → cross-entropy loss on 2-class logits
#   3. Backward pass → accumulate gradients for GRAD_ACCUM steps
#   4. Optimizer step → update weights with layer-wise LRs
#   5. Scheduler step → advance LR along warmup/decay curve
#   6. Evaluate on val set → compute_metrics → log accuracy + F1
#   7. Save checkpoint if val accuracy improved

trainer = Trainer(
    model           = model,
    args            = args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    optimizers      = (optimizer, scheduler),   # FIX 2: inject layer-wise LR decay
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
0,0.416000,0.396284,0.821275,0.820911
2,0.288400,0.435525,0.830130,0.830129
4,0.173300,0.567851,0.826024,0.826014


TrainOutput(global_step=126345, training_loss=0.3025152166340083, metrics={'train_runtime': 62102.9386, 'train_samples_per_second': 65.103, 'train_steps_per_second': 2.034, 'total_flos': 1.8839156968964751e+18, 'train_loss': 0.3025152166340083, 'epoch': 4.999901066503097})

## Cell 12 — Save Final Model

**What this cell does:** Saves the best model weights and the tokenizer together to a single directory.

**Why save both model and tokenizer?**
`from_pretrained()` expects both in the same directory. Saving them together produces a fully self-contained artefact — one folder, one load call, no need to specify the tokenizer separately.

**⚠️ Vast.ai reminder:** Cloud GPU instances have ephemeral disks. Stopping the instance permanently destroys `/workspace`. Always download checkpoints before terminating the session.

In [ ]:
# ── Cell 12: Save final model ──────────────────────────────────────────────
#
# PURPOSE: Persist the best model weights and its tokenizer together to disk.
#
# WHY SAVE BOTH MODEL AND TOKENIZER IN THE SAME DIRECTORY?
#   from_pretrained() expects both in the same folder. Saving them together
#   produces a self-contained artefact — one load call, no extra arguments needed.
#   trainer.save_model() → config.json + pytorch_model.bin
#   tokenizer.save_pretrained() → tokenizer_config.json + spm.model
#
# ⚠️ VAST.AI REMINDER:
#   Cloud GPU instances have ephemeral disks — stopping the instance permanently
#   destroys /workspace. Always download checkpoints before terminating.

trainer.save_model(FINAL_DIR)           # saves model weights + config
tokenizer.save_pretrained(FINAL_DIR)    # saves SentencePiece vocab + tokenizer config
print(f'Model + tokenizer saved to: {FINAL_DIR}')
print()
print('IMPORTANT: Download /workspace/checkpoints/ before stopping the instance!')
print('Terminal: tar -czf deberta_large_final.tar.gz /workspace/checkpoints/')


Model + tokenizer saved to: /workspace/checkpoints/deberta-large-sarcasm-final

IMPORTANT: Download /workspace/checkpoints/ before stopping the instance!
Terminal: tar -czf deberta_large_final.tar.gz /workspace/checkpoints/


## Cell 13 — Evaluate on Test Set

**What this cell does:** Produces a final, unbiased performance estimate on data the model has never seen during training or checkpoint selection.

**Why a separate test set?**
Early stopping selected the best checkpoint based on *validation* accuracy, so val accuracy is slightly optimistic (the selection process 'used' it). The test set was never involved in any decision — it gives the true expected performance on unseen Reddit comments.

**Reading the classification report:**
- *Precision*: of all examples predicted as class X, what fraction truly are X?
- *Recall*: of all true class X examples, what fraction did we correctly predict?
- *F1*: harmonic mean of precision and recall
- If precision ≈ recall ≈ F1 for both classes, the model is well-calibrated and not biased toward either class.

In [ ]:
# ── Cell 13: Evaluate on test set ──────────────────────────────────────────
#
# PURPOSE: Produce a final, unbiased performance estimate on data the model has
# never seen during training OR during checkpoint selection.
#
# WHY A SEPARATE TEST SET?
#   Early stopping selected the best checkpoint based on val accuracy, so val
#   accuracy is slightly optimistic (the selection process 'used' it indirectly).
#   The test set was never involved in any decision — it gives the true expected
#   performance on unseen Reddit comments.
#
# READING THE CLASSIFICATION REPORT:
#   precision : of all examples predicted class X, what fraction truly are X?
#   recall    : of all true class X examples, what fraction did we predict correctly?
#   f1-score  : harmonic mean of precision and recall
#   If precision ≈ recall ≈ f1 for both classes, the model is well-calibrated.

# trainer.predict() runs the model on test_dataset and collects logits + labels
predictions = trainer.predict(test_dataset)

# Convert 2-logit outputs → class index 0 or 1 via argmax
preds  = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print(f'\nTest Accuracy : {accuracy_score(labels, preds):.4f}')
print(f'Test F1 Score : {f1_score(labels, preds, average="weighted"):.4f}')
print('\nFull Classification Report:')
# target_names maps label indices (0, 1) to human-readable class names
print(classification_report(labels, preds, target_names=['not sarcastic', 'sarcastic']))



Test Accuracy : 0.8306
Test F1 Score : 0.8306

Full Classification Report:
               precision    recall  f1-score   support

not sarcastic       0.83      0.83      0.83     50541
    sarcastic       0.83      0.83      0.83     50537

     accuracy                           0.83    101078
    macro avg       0.83      0.83      0.83    101078
 weighted avg       0.83      0.83      0.83    101078



## Cell 14 — Package Model for Download

**What this cell does:** Compresses the entire checkpoints directory into a single `.tar.gz` archive for download from the Jupyter file browser.

**Why `tar` instead of `zip`?**
`tar -czf` preserves Unix file permissions and is faster on large collections of small files (PyTorch checkpoints consist of many files). The archive is also slightly smaller.

**`tar` flags:** `-c` (create), `-z` (gzip compress), `-f` (output file), `-C` (change to directory before archiving — produces relative internal paths, making the archive easier to extract anywhere).

In [ ]:
# ── Cell 14: Package model for download ────────────────────────────────────
#
# PURPOSE: Compress the entire checkpoints directory into a single .tar.gz file
# for easy download from the Jupyter file browser.
#
# WHY tar INSTEAD OF zip?
#   tar -czf preserves Unix file permissions and is faster on many small files
#   (PyTorch checkpoints = many small files). The archive is also slightly smaller.
#
# TAR FLAGS:
#   -c  : create a new archive
#   -z  : compress with gzip
#   -f  : write to the specified output file
#   -C  : change to this directory before archiving (produces relative internal
#         paths so the archive extracts cleanly anywhere)
#   '.' : archive everything in the -C directory

import subprocess
result = subprocess.run(
    ['tar', '-czf', '/workspace/deberta_large_sarcasm_final.tar.gz',
     '-C', '/workspace/checkpoints', '.'],
    capture_output=True, text=True
)

# Report archive size in MB (model is ~1.6 GB uncompressed)
size = os.path.getsize('/workspace/deberta_large_sarcasm_final.tar.gz') / 1e6
print(f'Archive: /workspace/deberta_large_sarcasm_final.tar.gz  ({size:.0f} MB)')
print('Download from the Jupyter file browser (left panel).')


Archive: /workspace/deberta_large_sarcasm_final.tar.gz  (23752 MB)
Download from the Jupyter file browser (left panel).
